### This is the Source Notebook to create what is required to develop CDC pipeline in Databricks

In [0]:
%sql
-- Drop bronze_table if it exists
DROP TABLE IF EXISTS bronze_table;
-- Drop silver_table if it exists
DROP TABLE IF EXISTS silver_table;
-- Drop gold_table if it exists
DROP TABLE IF EXISTS gold_table;



In [0]:
from pyspark.sql import functions as F

In [0]:
# Lets Create a Bronze Table with schema and data
bronze_data = [
    (1, 'Sai Aditya', 284.34, 2026),
    (2, 'Rachel', 658.26, 2026),
    (3, 'Allen', 465.68, 2026),
    (4, 'Brooke', 349.34, 2026),
    (5, 'Charlie', 567.34, 2026)
]

bronze_df = spark.createDataFrame(
    bronze_data,
    ['id', 'name', 'amount', 'year']
).withColumn(
    "ingested_at",
    F.date_format(
        F.from_utc_timestamp(
            F.current_timestamp(),
            "America/Chicago"
        ),
        "MM/dd/yyyy hh:mm:ss a"
    )
)

bronze_df.write.format("delta").mode("overwrite").saveAsTable("bronze_table")

In [0]:
# Lets Create a Silver Table with schema and data
silver_data = [
    (1, 'Sai Aditya', 284.34, 2026),
    (2, 'Rachel', 658.26, 2026),
    (3, 'Allen', 465.68, 2026),
    (4, 'Brooke', 349.34, 2026),
    (5, 'Charlie', 567.34, 2026)
]

silver_df = spark.createDataFrame(
    silver_data ,
    ['id', 'name', 'amount', 'year']
).withColumn(
    "ingested_at",
    F.date_format(
        F.from_utc_timestamp(
            F.current_timestamp(),
            "America/Chicago"
        ),
        "MM/dd/yyyy hh:mm:ss a"
    )
)

silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_table")

In [0]:
# Lets Create a Gold Table with Aggrgated data for each year with Rank 

gold_df = spark.sql("""
    WITH aggregated AS (
        SELECT 
            name,
            year,
            SUM(amount) AS total_amount
        FROM silver_table
        GROUP BY name, year
    )
    SELECT 
        name,
        year,
        total_amount,
        DENSE_RANK() OVER (PARTITION BY year ORDER BY total_amount DESC) AS rank
    FROM aggregated
    ORDER BY year, rank ASC
""")

final_df_gold = gold_df.withColumn(
    "ingested_at",
    F.date_format(
        F.from_utc_timestamp(
            F.current_timestamp(),
            "America/Chicago"
        ),
        "MM/dd/yyyy hh:mm:ss a"
    )
)

final_df_gold.write.mode("overwrite").saveAsTable("gold_table")

In [0]:
# Now enable delta change enable for bronze table
spark.sql("ALTER TABLE bronze_table SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")

Check wheter CDF is enabled or not 

In [0]:
%sql
DESCRIBE EXTENDED bronze_table;